# Estructura del proyecto

## Lectura de datos
El programa inicia leyendo un archivo CSV que contiene la información de la encuesta.  
Para esto se utiliza la función `leer_datos`, que abre el archivo, recorre cada fila y la procesa.

Aquí es donde se obtienen los datos “en bruto” (tal como vienen en el archivo).


## Transformación de datos
Cada fila del archivo CSV se transforma en una estructura más organizada usando la función `recorrer_datos`.

En esta etapa:
- Se agrupan los datos en secciones (preferencias, consumo, experiencia, nps).
- Se convierten los tipos de datos (texto a números, texto a booleanos).
  
Esto permite trabajar con la información de forma más clara y ordenada.


## Modelo de datos (Clase Encuestado)
Se utiliza la clase `Encuestado` para representar a cada persona de la encuesta como un objeto.

Esto significa que:
- Cada encuestado tiene atributos propios (comida, gasto, satisfacción, etc.).
- Ya no trabajamos con diccionarios complicados, sino con objetos fáciles de usar.

Esto mejora la organización del código y facilita el acceso a los datos.


## Almacenamiento de datos
Todos los objetos `Encuestado` se guardan en una lista.

Esta lista:
- Contiene todos los registros (por ejemplo, los 20,000 datos).
- Es la base para realizar todos los reportes.


## Procesamiento de datos
Una vez cargados los datos, se utilizan métodos (en otra clase como `AnalizadorEncuesta`) para analizarlos.

Aquí es donde se:
- Hacen cálculos
- Se agrupan datos
- Se generan estadísticas


## Presentación de resultados
Finalmente, los resultados de los reportes se muestran en pantalla usando `print`.

Esto permite ver:
- Promedios
- Conteos
- Relaciones entre datos


## Conclusión
El proyecto está organizado en etapas claras:
1. Leer datos
2. Transformarlos
3. Guardarlos en objetos
4. Procesarlos
5. Mostrar resultados

Esta estructura hace que el programa sea más ordenado, fácil de entender y fácil de mantener.

In [13]:
import csv

# Clase para guardar los datos de cada encuestado
class Encuestado:
    # Constructor de la clase (se ejecuta cuando creamos un objeto)
    def __init__(self, d):
        self.id = d["id"]
        self.comida_preferida = d["preferencias"]["comida_preferida"]
        self.frecuencia = d["preferencias"]["frecuencia"]
        self.gasto = d["consumo"]["gasto"]
        self.producto = d["experiencia"]["producto"]
        self.servicio = d["experiencia"]["servicio"]
        self.recomendacion = d["nps"]["recomendacion"]
        self.calificacion_general = d["nps"]["general"]
        self.precio = d["experiencia"]["precio"]
        self.tiempo_entrega = d["experiencia"]["tiempo de entrega"] 
        self.volveria = d["nps"]["volveria"]

# Función para transformar cada fila del CSV en un diccionario organizado
def recorrer_datos(fila):
    # Retornamos un diccionario con estructura ordenada
    return {
        # Convertimos el id a entero
        "id": int(fila["id"]),
        
        # Agrupamos datos de preferencias
        "preferencias": {
            "comida_preferida": fila["comida_preferida"],  # texto
            "frecuencia": fila["frecuencia_consumo"],      # texto
        },
        
        # Agrupamos datos de consumo
        "consumo": {
            "gasto": float(fila["gasto_promedio"])  # convertimos a decimal
        },
        
        # Agrupamos datos de experiencia
        "experiencia": {
            "producto": int(fila["satisfaccion_producto"]),  # entero
            "servicio": int(fila["satisfaccion_servicio"]),  # entero
            "tiempo de entrega": fila["tiempo_entrega"],     # texto
            "precio": fila["precio_percepcion"],             # texto
        },
        
        # Agrupamos datos de NPS
        "nps": {
            "recomendacion": int(fila["recomendaria_nps"]),  # entero
            
            # Convertimos "Sí"/"No" a True o False
            "volveria": fila["volveria_comprar"].strip().lower() in ["sí", "si"],
            
            "general": int(fila["calificacion_general"]) # entero
        }
    }

# Función para leer el archivo CSV
def leer_datos(nombre_archivo):
    # Uso de listas para almacenar los 20,000 registros.
    lista_encuestados = []
    
    try:
        # Abrimos el archivo en modo lectura con codificación latin-1
        with open(nombre_archivo, mode='r', encoding='latin-1') as archivo:
            
            # DictReader convierte cada fila en un diccionario automáticamente
            lector = csv.DictReader(archivo)

            # Recorremos cada fila del archivo
            for fila in lector:

                # Transformamos la fila en un diccionario estructurado
                registro = recorrer_datos(fila)

                # Creamos un objeto de tipo Encuestado usando ese diccionario
                encuestado = Encuestado(registro)

                # Guardamos el objeto en la lista
                lista_encuestados.append(encuestado)

    # Si el archivo no existe, mostramos un error
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {nombre_archivo}")
    
    # Retornamos la lista de encuestados ya procesada
    return lista_encuestados

#print(len(datos))

# Reporte 1. Total de encuestados
### ¿Qué hace?
Este reporte calcula la **cantidad total de personas que respondieron la encuesta**.

### ¿Cómo lo hace línea por línea?
- Accede a la lista de encuestados almacenada en la clase.
- Usa la función `len()` para contar cuántos elementos hay en la lista.
- Retorna ese número como resultado final.

In [ ]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 1)
# ================================
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main
    
    def total_encuestados(self):
        return len(self.encuestados)

# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 1)
# ================================
    
from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)
analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos[cite: 3]

print("1. Total de encuestados:", analizador.total_encuestados())

1. Total de encuestados: 4


## Reporte 2. Comida más preferida

### ¿Qué hace?
Este reporte identifica cuál es la **comida más elegida por los encuestados**.

### ¿Cómo lo hace línea por línea?
- Recorre todos los encuestados y guarda sus comidas preferidas en una lista.
- Usa una función para contar cuántas veces aparece cada comida.
- Obtiene la comida con mayor cantidad de repeticiones.
- Retorna esa comida como la más preferida.

In [16]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 2)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    # Función auxiliar para contar elementos
    def _contar_elementos(self, lista):
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def comida_mas_preferida(self):
        lista = [e.comida_preferida for e in self.encuestados]
        conteo = self._contar_elementos(lista)
        return max(conteo, key=conteo.get)


# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 2)
# ================================

from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)
analizador = AnalizadorEncuesta(datos)

print("2. Comida más preferida:", analizador.comida_mas_preferida())

2. Comida más preferida: Pizza


## Reporte 3. Cantidad por tipo de comida

### ¿Qué hace?
Este reporte muestra **cuántas personas prefieren cada tipo de comida**.

### ¿Cómo lo hace línea por línea?
- Recorre la lista de encuestados.
- Extrae la comida preferida de cada uno.
- Usa un diccionario para contar cuántas veces aparece cada comida.
- Retorna el diccionario con los resultados.

In [17]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 3)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    def _contar_elementos(self, lista):
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def cantidad_por_comida(self):
        lista = [e.comida_preferida for e in self.encuestados]
        return self._contar_elementos(lista)


# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 3)
# ================================

from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)
analizador = AnalizadorEncuesta(datos)

print("3. Cantidad por tipo de comida:")
for comida, total in analizador.cantidad_por_comida().items():
    print(f"   - {comida}: {total}")

3. Cantidad por tipo de comida:
   - Pollo Frito: 1
   - Pizza: 2
   - Hot Dogs: 1


## Reporte 4. Frecuencia de consumo más común

### ¿Qué hace?
Este reporte identifica cuál es la **frecuencia de consumo más repetida** (por ejemplo: diario, semanal, etc.).

### ¿Cómo lo hace línea por línea?
- Recorre todos los encuestados.
- Extrae la frecuencia de consumo de cada uno.
- Cuenta cuántas veces aparece cada frecuencia.
- Obtiene la frecuencia que más se repite.
- Retorna ese valor.

In [18]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 4)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    def _contar_elementos(self, lista):
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def frecuencia_mas_comun(self):
        lista = [e.frecuencia for e in self.encuestados]
        conteo = self._contar_elementos(lista)
        return max(conteo, key=conteo.get)


# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 4)
# ================================

from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)
analizador = AnalizadorEncuesta(datos)

print("4. Frecuencia de consumo más común:", analizador.frecuencia_mas_comun())

4. Frecuencia de consumo más común: Diario


## Reporte 5. Promedio de gasto general

### ¿Qué hace?
Este reporte calcula el **gasto promedio de todos los encuestados**.

### ¿Cómo lo hace línea por línea?
- Recorre todos los encuestados.
- Suma el gasto de cada uno.
- Divide la suma total entre la cantidad de encuestados.
- Retorna el promedio de gasto.

In [19]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 5)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    def promedio_gasto_general(self):
        total = sum(e.gasto for e in self.encuestados)
        return total / len(self.encuestados)


# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 5)
# ================================

from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)
analizador = AnalizadorEncuesta(datos)

print(f"5. Promedio de gasto general: ${analizador.promedio_gasto_general():.2f}")

5. Promedio de gasto general: $104.00


In [20]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

# Reporte 6

# Lo que se puso en practica:
- Utilizamos propiedades como .gasto o .comida_preferida

- Utilizamos listas para almacenar los 20,000 registros y diccionarios para realizar los conteos y agrupaciones

In [43]:
# Primero debemos importar y leer, sino 'datos' no existe
from lectura_datos import leer_datos
datos = leer_datos("prueba.csv")

# Funcion reporte 6
def reporte_6(encuestados):
    # Creamos diccionarios vacíos para ir "acumulando"
    acum, cont = {}, {}
    
    for e in encuestados:
        # Buscamos la comida y sumamos el gasto
        # .get(..., 0) evita errores si la comida aún no está en la lista
        acum[e.comida_preferida] = acum.get(e.comida_preferida, 0) + e.gasto
        # Contamos a la persona para promediar después
        cont[e.comida_preferida] = cont.get(e.comida_preferida, 0) + 1
    
    # Retornamos el promedio (Suma / Cantidad) usando un diccionario por comprensión
    return {c: acum[c] / cont[c] for c in acum}

# Resultado 
print("Reporte 6. Promedio de gasto por comida:")
for comida, promedio in reporte_6(datos).items():
    print(f"   - {comida}: ${promedio:.2f}")

Reporte 6. Promedio de gasto por comida:
   - Pollo Frito: $70.00
   - Pizza: $137.50
   - Hot Dogs: $71.00


FUNCIONAMIENTO: Busca en la lista a cada encuestado para ver qué comió y cuánto pagó, con el diccionario
vamos a recopilar los datos para sumarlo y contar a las personas por el grupo de comida, y luego lo dividimos
para sacar el promedio del dinero según el grupo de comida.

# Reporte 7

# Lo que se puso en practica:
- Usamos la propiedad .producto en vez de buscar en una lista de textos

- Un bucle for que va de  objeto en objeto para extraer solo la calificación del producto.

In [44]:
# Primero debemos importar y leer, sino 'datos' no existe
from lectura_datos import leer_datos
datos = leer_datos("prueba.csv")

# Funcion de reporte 7
def reporte_7(encuestados):
    # Sumamos todos los puntajes de satisfacción del producto y dividimos entre el total
    return sum(e.producto for e in encuestados) / len(encuestados)

# Resultado
print(f"Reporte 7: Satisfacción promedio producto: {reporte_7(datos):.2f} / 10")

Reporte 7: Satisfacción promedio producto: 3.75 / 10


FUNCIONAMIENTO: el programa recorre toda nuestra lista de encuestados y va sumando las calificaciones que le dieron a la comida (el producto). Al final, toma esa suma total y la divide entre la cantidad de personas que respondieron (usando len).

# Reporte 8

# Lo que se puso en practica:
- Usamos el atributo .servicio para guardar la nota del cliente que le puso a la atencion que recibio

- Usamos sum() para totalizar los puntos y len() para saber entre cuántas personas dividir.

- Utilizamos el metodo reporte 8 para que guarde la funcion de hacer el proceso que le pedimos

In [45]:
# Primero debemos importar y leer, sino 'datos' no existe
from lectura_datos import leer_datos
datos = leer_datos("prueba.csv")

# Funcion del reporte 8
def reporte_8(encuestados):
    # Sumamos los puntajes de la columna 'servicio' y promediamos
    return sum(e.servicio for e in encuestados) / len(encuestados)

# Resultado
print(f" Reporte 8: Satisfacción promedio servicio: {reporte_8(datos):.2f} / 10")

 Reporte 8: Satisfacción promedio servicio: 4.75 / 10


FUNCIONAMIENTO: Igual que el anterior vamos sumando todas las calificaciones que los clientes dieron específicamente al servicio al cliente. Se toma esa gran suma y la divide entre el total de encuestados para darnos el  promedio.

# Reporte 9

# Lo que se puso en practica:
- Usamos el atributo .calificacion_general para acceder a las calificacion generales de la encuestas.

- Usamos sum() para acumular las notas y len() para obtener la division y sacar el promedio.

- Utilizamos el metodo reporte 9 para que guarde la funcion de hacer el proceso que le pedimos

In [46]:
# Primero debemos importar y leer, sino 'datos' no existe
from lectura_datos import leer_datos
datos = leer_datos("prueba.csv")

# Funcion del reporte 9
def reporte_9(encuestados):
    # Sumamos la columna 'calificacion_general' y sacamos el promedio
    return sum(e.calificacion_general for e in encuestados) / len(encuestados)

# Resultado
print(f"Reporte 9: Calificación general promedio: {reporte_9(datos):.2f} / 10")

Reporte 9: Calificación general promedio: 4.25 / 10


FUNCIONAMIENTO: Entramos a cada encuesta y toma la nota final que el cliente le dio al restaurante. Suma todas esas notas y las divide entre el total de personas

# Reporte 10

# Lo que se puso en practica:
- Utilizamos el objeto conteo = {} donde se ingresaran los datos del conteo

- Utilizamos la propiedad .precio para sacar el conteo

- Método .get(e.precio, 0) donde se vera si ya existe y sumarle 1 al total.

- Usamos .items() para recorrer los resultados y poder imprimirlos.

In [25]:
# Primero debemos importar y leer, sino 'datos' no existe
from lectura_datos import leer_datos
datos = leer_datos("prueba.csv")

#Funcion del reporte 10
def reporte_10(encuestados):
    # Creamos un diccionario para contar las frecuencias
    conteo = {}
    for e in encuestados:
        # Usamos la propiedad .precio como clave y sumamos 1 cada vez que aparece
        conteo[e.precio] = conteo.get(e.precio, 0) + 1
    return conteo

# Resultado
print("10. Distribución de percepción de precios:")
resultados_10 = reporte_10(datos)
for nivel, total in resultados_10.items():
    print(f"    > {nivel}: {total} encuestados")

10. Distribución de percepción de precios:
    > Bajo: 1 encuestados
    > Medio: 2 encuestados
    > Alto: 1 encuestados


FUNCIONAMIENTO: Se recorre la lista de objetos y usar la propiedad .precio ,  cada vez que el programa encuentra una respuesta, se ira sumando al  contador de esa categoría específica y al finalizar el ciclo, se tiene un resumen total de cuántas personas pertenecen a cada nivel de precio.

# Estructura Interna del Sistema de Análisis

## El Constructor (`__init__`)
Cuando "contratamos" a este gerente (es decir, cuando creamos el objeto), le entregamos una gran carpeta con todas las encuestas (`lista_objetos`).  

El constructor se encarga de:
- Recibir esa información inicial  
- Guardarla en una variable interna llamada `self.encuestados`  

De esta manera, todos los métodos del programa pueden acceder a los datos sin tener que volver a cargarlos.  


## Funciones 
El sistema cuenta con dos funciones auxiliares clave (identificadas con `_` al inicio, lo que indica uso interno).  
Estas funciones evitan repetir código innecesariamente.  


### 1. `_contar_elementos`

#### ¿Qué hace?
Cuenta cuántas veces aparece cada elemento dentro de una lista.

#### ¿Cómo funciona?
- Recibe una lista (por ejemplo: tipos de comida, tiempos de entrega, etc.)  
- Recorre cada elemento  
- Usa un diccionario para llevar el conteo  
- Devuelve un resumen con las frecuencias  

# Reporte 11: Distribución de tiempo de entrega (distribucion_tiempo_entrega)

## ¿Qué hace?
Nos dice cuántas personas eligieron cada opción de tiempo de entrega (por ejemplo, cuántos dijeron "Rápido" y cuántos "Lento").

## ¿Cómo lo hace?
- Crea una lista rápida (`tiempos = [...]`) extrayendo únicamente la opinión del tiempo de entrega de cada encuestado.  
- Pasa esta lista a la función auxiliar `_contar_elementos`.  
- La función se encarga de contar automáticamente la frecuencia de cada categoría y devuelve el resultado listo. 

In [35]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

    # Reporte 11. Distribución de tiempo de entrega.
    def distribucion_tiempo_entrega(self):
        """11. Distribución de tiempo de entrega"""
        # Extraemos solo los tiempos en una lista y se los pasamos a _contar_elementos
        tiempos = [e.tiempo_entrega for e in self.encuestados]
        return self._contar_elementos(tiempos)

from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))

analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos
# Reporte 11
print(f"\n11. Distribución de tiempo de entrega:")
for tiempo, total in analizador.distribucion_tiempo_entrega().items():
    print(f"   - {tiempo}: {total} encuestados")


11. Distribución de tiempo de entrega:
   - Aceptable: 2 encuestados
   - Rpido: 1 encuestados
   - Lento: 1 encuestados


# Reporte 12: Porcentaje de clientes que volverían a comprar

## ¿Qué hace?
Calcula la lealtad básica, mostrando el porcentaje de personas que dijeron que sí regresarían al restaurante.

## ¿Cómo lo hace (línea por línea)?
- Verifica que la lista de encuestados no esté vacía (`if not self.encuestados: return 0`).  
- Utiliza una expresión eficiente:  
  `sum(1 for e in encuestados if e.volveria)` para contar solo los valores verdaderos.  
- Divide el total de respuestas positivas entre el total de encuestados.  
- Multiplica por 100 para obtener el porcentaje final.  

In [37]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

    # Reporte 12. Porcentaje de clientes que volverían a comprar. 
    def porcentaje_volverian_comprar(self):
        """12. Porcentaje de clientes que volverían a comprar"""
        if not self.encuestados: return 0
        # Sumamos 1 solo si e.volveria es verdadero
        votos_si = sum(1 for e in self.encuestados if e.volveria)
        return (votos_si / len(self.encuestados)) * 100

from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))

analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos

# Reporte 12
print(f"\n12. Porcentaje de clientes que volverían a comprar: {analizador.porcentaje_volverian_comprar():.2f}%")


12. Porcentaje de clientes que volverían a comprar: 50.00%


# Reporte 13: Cálculo del NPS general (calculo_nps_general)

## ¿Qué hace?
Calcula el indicador global de recomendación (Net Promoter Score) de todos los clientes.

## ¿Cómo lo hace (línea por línea)?
- Extrae las calificaciones de recomendación (0 a 10) en una lista llamada `calificaciones`.  
- Pasa esta lista a la función `_calcular_nps`.  
- La función aplica la fórmula del NPS y devuelve el resultado final.

In [38]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

    # Reporte 13. Cálculo del NPS general. 
    def calculo_nps_general(self):
        """13. Cálculo del NPS general"""
        # Extraemos todas las calificaciones de recomendación y se las pasamos a _calcular_nps
        calificaciones = [e.recomendacion for e in self.encuestados]
        return self._calcular_nps(calificaciones)

from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))

analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos

# Reporte 13
print(f"\n13. Cálculo del NPS general: {analizador.calculo_nps_general():.2f}")


13. Cálculo del NPS general: -75.00


# Reporte 14: Cantidad de promotores, pasivos y detractores (conteo_promotores_pasivos_detractores)

## ¿Qué hace?
Clasifica a todos los clientes según su nivel de lealtad.

## ¿Cómo lo hace (línea por línea)?
- Extrae todas las calificaciones de recomendación en la lista `calificaciones`.  
- Crea un diccionario con tres categorías: `Promotores`, `Pasivos` y `Detractores`.  
- Utiliza conteo condicional:
  - **Promotores:** calificaciones ≥ 9  
  - **Pasivos:** calificaciones entre 7 y 8  
  - **Detractores:** calificaciones ≤ 6  
- Devuelve el diccionario con los totales. 

In [39]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

    # Reporte 14. Cantidad de promotores, pasivos y detractores. 
    def conteo_promotores_pasivos_detractores(self):
        """14. Cantidad de promotores, pasivos y detractores"""
        calificaciones = [e.recomendacion for e in self.encuestados]
        return {
            "Promotores": sum(1 for n in calificaciones if n >= 9),
            "Pasivos": sum(1 for n in calificaciones if 7 <= n <= 8),
            "Detractores": sum(1 for n in calificaciones if n <= 6)
        }

from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))

analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos
# Reporte 14
print(f"\n14. Cantidad de promotores, pasivos y detractores:")
for segmento, cantidad in analizador.conteo_promotores_pasivos_detractores().items():
    print(f"   - {segmento}: {cantidad} clientes")


14. Cantidad de promotores, pasivos y detractores:
   - Promotores: 0 clientes
   - Pasivos: 1 clientes
   - Detractores: 3 clientes


# Reporte 15: NPS por tipo de comida (nps_por_tipo_comida)

## ¿Qué hace?
Determina el nivel de recomendación (NPS) para cada tipo de comida de forma individual.

## ¿Cómo lo hace (línea por línea)?
- Crea un diccionario vacío llamado `grupos`.  
- Recorre cada encuestado y utiliza `.setdefault()`:
  - Si la comida no existe como clave, crea una lista vacía.  
  - Agrega la calificación de recomendación a la lista correspondiente.  
- Crea un nuevo diccionario `nps_comidas`.  
- Recorre cada tipo de comida con sus calificaciones agrupadas.  
- Aplica la función `_calcular_nps` a cada grupo.  
- Guarda el resultado del NPS por cada comida.  

In [40]:
#Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    def _contar_elementos(self, lista):
        """Recibe cualquier lista y devuelve un diccionario con el conteo."""
        conteo = {}
        for item in lista:
            conteo[item] = conteo.get(item, 0) + 1
        return conteo

    def _calcular_nps(self, lista_calificaciones):
        """Recibe una lista de lista_calificaciones (0-10) y aplica la fórmula NPS."""
        if not lista_calificaciones: return 0
        # suma y cuenta las calificaciones
        promotores = sum(1 for n in lista_calificaciones if n >= 9)
        detractores = sum(1 for n in lista_calificaciones if n <= 6)
        return ((promotores - detractores) / len(lista_calificaciones)) * 100

    # Reporte 15. NPS por tipo de comida. 
    def nps_por_tipo_comida(self):
        """15. NPS por tipo de comida"""
        # 1. Agrupamos las calificaciones de recomendación separadas por comida
        grupos = {}
        for e in self.encuestados:
            # .setdefault() crea la lista si no existe, y luego agrega la calificacion
            grupos.setdefault(e.comida_preferida, []).append(e.recomendacion)
            
        # 2. Reutilizamos _calcular_nps para calcular el NPS para cada grupo
        nps_comidas = {}
        for comida, calificaciones in grupos.items():
            nps_comidas[comida] = self._calcular_nps(calificaciones)
            
        return nps_comidas

from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))

analizador = AnalizadorEncuesta(datos) # 'datos' contiene lo que leyó leer_datos
# Reporte 15
print(f"\n15. NPS por tipo de comida:")
for comida, nps in analizador.nps_por_tipo_comida().items():
    print(f"   - {comida}: {nps:.2f} NPS")


15. NPS por tipo de comida:
   - Pollo Frito: -100.00 NPS
   - Pizza: -50.00 NPS
   - Hot Dogs: -100.00 NPS


## Reporte 16. Comida con mejor satisfacción promedio

### ¿Qué hace?
Este reporte identifica cuál es la comida que tiene el **mayor nivel de satisfacción promedio**, considerando tanto la calificación del producto como del servicio.

### ¿Cómo lo hace línea por línea?
- Crea un diccionario para guardar cada comida con su suma de satisfacción y cantidad de personas.
- Recorre todos los encuestados uno por uno.
- Obtiene la comida preferida de cada persona.
- Calcula su satisfacción promedio (producto + servicio) / 2.
- Si la comida no está en el diccionario, la agrega con valores iniciales.
- Suma la satisfacción y aumenta el contador.
- Luego recorre el diccionario para calcular el promedio por comida.
- Compara los promedios y guarda el más alto.
- Retorna la comida con mejor satisfacción.

In [8]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 16)
# ================================

# Definimos la clase que se encargará de procesar toda la información
class AnalizadorEncuesta:
    # El constructor recibe la lista de objetos que creamos al leer el CSV
    def __init__(self, lista_objetos):
        
        self.encuestados = lista_objetos # Esta es la lista 'datos' del main

    # Reporte 16. Comida con mejor satisfacción promedio.
    def comida_mejor_satisfaccion(self):
    
        datos_comidas = {}  # Diccionario para agrupar
    
        # Recorremos los objetos
        for p in self.encuestados:
        
            comida = p.comida_preferida
        
            # Promedio de satisfacción
            satisfaccion = (p.producto + p.servicio) / 2
        
            # Si no existe la comida
            if comida not in datos_comidas:
                datos_comidas[comida] = [0, 0]  # suma, cantidad
        
            # Acumulamos
            datos_comidas[comida][0] += satisfaccion
            datos_comidas[comida][1] += 1
    
        # Buscar mejor
        mejor_comida = ""
        mejor_promedio = 0
    
        for comida, valores in datos_comidas.items():
            promedio = valores[0] / valores[1]
        
            if promedio > mejor_promedio:
                mejor_promedio = promedio
                mejor_comida = comida
    
        return mejor_comida


# ================================
# MODULO: PRESENTACIÓN (IMPRESIÓN DEL REPORTE 16)
# ================================
from lectura_datos import *
from reportes import *

# 1. Cargar datos al inicio
ruta = "prueba.csv"
datos = leer_datos(ruta)

#print(len(datos))
# Crear objeto analizador (datos debe venir del CSV)
analizador = AnalizadorEncuesta(datos)

# Imprimiendo resultados
print("\n16. Comida con mejor satisfacción promedio")
print(analizador.comida_mejor_satisfaccion())


16. Comida con mejor satisfacción promedio
Pizza


## Reporte 17. Comida con menor satisfacción promedio

### ¿Qué hace?
Este reporte encuentra la comida que tiene el **nivel más bajo de satisfacción promedio**.

### ¿Cómo lo hace línea por línea?
- Crea un diccionario para agrupar las comidas.
- Recorre todos los encuestados.
- Obtiene la comida y la satisfacción promedio de cada persona.
- Si la comida no existe en el diccionario, la inicializa.
- Acumula la suma de satisfacción y el número de personas.
- Usa una variable con valor infinito para encontrar el menor promedio.
- Recorre el diccionario y calcula el promedio de cada comida.
- Compara y guarda el valor más bajo.
- Retorna la comida con menor satisfacción.

In [9]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 17)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    # Reporte 17. Comida con menor satisfacción promedio.
    def comida_menor_satisfaccion(self):
    
        datos_comidas = {}
        
        # Recorremos los encuestados
        for p in self.encuestados:
            
            comida = p.comida_preferida
            satisfaccion = (p.producto + p.servicio) / 2
            
            if comida not in datos_comidas:
                datos_comidas[comida] = [0, 0]
            
            datos_comidas[comida][0] += satisfaccion
            datos_comidas[comida][1] += 1
        
        peor_comida = ""
        peor_promedio = float("inf")
        
        for comida, valores in datos_comidas.items():
            promedio = valores[0] / valores[1]
            
            if promedio < peor_promedio:
                peor_promedio = promedio
                peor_comida = comida
        
        return peor_comida


# ================================
# MODULO: PRESENTACIÓN
# ================================
from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)

analizador = AnalizadorEncuesta(datos)

print("\n17. Comida con menor satisfacción promedio")
print(analizador.comida_menor_satisfaccion())


17. Comida con menor satisfacción promedio
Pizza


## Reporte 18. Relación entre precio percibido y recomendación

### ¿Qué hace?
Este reporte analiza **cómo influye el precio percibido en la recomendación** que hacen los clientes.

### ¿Cómo lo hace línea por línea?
- Crea un diccionario para agrupar por tipo de precio.
- Recorre todos los encuestados.
- Obtiene el precio percibido y la calificación de recomendación.
- Si el precio no está en el diccionario, lo inicializa.
- Suma las recomendaciones y cuenta las personas.
- Luego recorre el diccionario para calcular el promedio por precio.
- Guarda esos promedios en un nuevo diccionario.
- Retorna la relación entre precio y recomendación.

In [10]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 18)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    # Reporte 18. Relación entre precio percibido y recomendación.
    def relacion_precio_recomendacion(self):
        
        datos = {}
        
        for e in self.encuestados:
            precio = e.precio
            recomendacion = e.recomendacion
            
            if precio not in datos:
                datos[precio] = [0, 0]
            
            datos[precio][0] += recomendacion
            datos[precio][1] += 1
        
        resultado = {}
        
        for precio, valores in datos.items():
            resultado[precio] = valores[0] / valores[1]
        
        return resultado


# ================================
# MODULO: PRESENTACIÓN
# ================================
from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)

analizador = AnalizadorEncuesta(datos)

print("\n18. Relación entre precio y recomendación")
for precio, promedio in analizador.relacion_precio_recomendacion().items():
    print(f"   - {precio}: {promedio:.2f}")


18. Relación entre precio y recomendación
   - Bajo: 3.00
   - Medio: 6.50
   - Alto: 1.00


## Reporte 19. Relación entre tiempo de entrega y satisfacción

### ¿Qué hace?
Este reporte analiza **cómo el tiempo de entrega afecta la satisfacción del cliente**.

### ¿Cómo lo hace línea por línea?
- Crea un diccionario para agrupar por tiempo de entrega.
- Recorre todos los encuestados.
- Obtiene el tiempo de entrega y calcula la satisfacción promedio.
- Si el tiempo no existe en el diccionario, lo inicializa.
- Acumula la satisfacción y el número de personas.
- Recorre el diccionario para calcular el promedio por tiempo.
- Guarda los resultados en un nuevo diccionario.
- Retorna la relación entre tiempo y satisfacción.

In [11]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 19)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    # Reporte 19. Relación entre tiempo de entrega y satisfacción.
    def relacion_tiempo_satisfaccion(self):
        
        datos = {}
        
        for e in self.encuestados:
            tiempo = e.tiempo_entrega
            satisfaccion = (e.producto + e.servicio) / 2
            
            if tiempo not in datos:
                datos[tiempo] = [0, 0]
            
            datos[tiempo][0] += satisfaccion
            datos[tiempo][1] += 1
        
        resultado = {}
        
        for tiempo, valores in datos.items():
            resultado[tiempo] = valores[0] / valores[1]
        
        return resultado


# ================================
# MODULO: PRESENTACIÓN
# ================================
from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)

analizador = AnalizadorEncuesta(datos)

print("\n19. Relación entre tiempo de entrega y satisfacción")
for tiempo, promedio in analizador.relacion_tiempo_satisfaccion().items():
    print(f"   - {tiempo}: {promedio:.2f}")


19. Relación entre tiempo de entrega y satisfacción
   - Aceptable: 4.75
   - Rï¿½pido: 2.00
   - Lento: 5.50


## Reporte 20. Perfil predominante del consumidor

### ¿Qué hace?
Este reporte identifica el **perfil más común del consumidor**, es decir:
- comida más elegida
- frecuencia más común
- percepción de precio más común

### ¿Cómo lo hace línea por línea?
- Crea tres diccionarios para contar comidas, frecuencias y precios.
- Recorre todos los encuestados.
- Cuenta cuántas veces aparece cada comida.
- Cuenta cuántas veces aparece cada frecuencia.
- Cuenta cuántas veces aparece cada percepción de precio.
- Usa `max()` para encontrar el valor más repetido en cada categoría.
- Guarda esos valores como el perfil predominante.
- Retorna un diccionario con los resultados.

In [12]:
# ================================
# MODULO: PROCESAMIENTO (LÓGICA DEL REPORTE 20)
# ================================

class AnalizadorEncuesta:
    def __init__(self, lista_objetos):
        self.encuestados = lista_objetos

    # Reporte 20. Perfil predominante del consumidor.
    def perfil_predominante(self):
        
        comidas = {}
        frecuencias = {}
        precios = {}
        
        for e in self.encuestados:
            
            comidas[e.comida_preferida] = comidas.get(e.comida_preferida, 0) + 1
            frecuencias[e.frecuencia] = frecuencias.get(e.frecuencia, 0) + 1
            precios[e.precio] = precios.get(e.precio, 0) + 1
        
        comida_top = max(comidas, key=comidas.get)
        frecuencia_top = max(frecuencias, key=frecuencias.get)
        precio_top = max(precios, key=precios.get)
        
        return {
            "comida_preferida": comida_top,
            "frecuencia": frecuencia_top,
            "precio_percibido": precio_top
        }


# ================================
# MODULO: PRESENTACIÓN
# ================================
from lectura_datos import *

ruta = "prueba.csv"
datos = leer_datos(ruta)

analizador = AnalizadorEncuesta(datos)

print("\n20. Perfil predominante del consumidor")
perfil = analizador.perfil_predominante()

for clave, valor in perfil.items():
    print(f"   - {clave}: {valor}")


20. Perfil predominante del consumidor
   - comida_preferida: Pizza
   - frecuencia: Diario
   - precio_percibido: Medio
